# Validación ERA5-Land: CDS frente a Earth Engine

Compara unas 30 horas sobre exactamente los mismos puntos de la cuadrícula. Primero contrasta variables brutas y después las variables derivadas del proyecto. CDS y Earth Engine se ejecutan secuencialmente para compartir fechas, geometría y criterios; dos notebooks paralelos podrían comparar muestras distintas y competir por recursos.

## 1. Configuración y credenciales

In [ ]:
from pathlib import Path
import os

# Completa únicamente estos dos campos. Earth Engine usa la autorización local ya guardada.
EE_PROJECT = "tfm-gnn"  # ID del proyecto de Google Earth Engine
CDS_API_KEY = os.getenv("CDS_API_KEY", "")  # Token personal de Copernicus CDS

# Si se dejan vacíos, también pueden proceder de variables de entorno.
EE_PROJECT = EE_PROJECT or os.getenv("EE_PROJECT", "")
CDS_API_KEY = CDS_API_KEY or os.getenv("CDS_API_KEY", "")

# Muestra corta y editable: 60 horas evaluadas y tres días previos de calentamiento.
SAMPLE_START_UTC = "2020-10-14T00:00:00Z"
N_HOURS = 60
CENTER_LAT = 37.3554728
CENTER_LON = -5.9808173
AREA_PADDING_DEG = 0.15
OVERWRITE = True  # Mantener True para medir una descarga CDS real, no una lectura de caché.

OUTPUT_DIR = Path.cwd() / "validation_era5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Preparación de la muestra común

In [ ]:
import json
from time import perf_counter, sleep
from collections import defaultdict

import cdsapi
import ee
import numpy as np
import pandas as pd
import xarray as xr

from tools.scientific_variables import deaccumulate_era5_land, derive_era5

if not EE_PROJECT:
    raise ValueError("Completa EE_PROJECT en la cabecera.")
if not CDS_API_KEY:
    raise ValueError("Completa CDS_API_KEY en la cabecera.")

try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    print("Earth Engine necesita autorización. Se abrirá el navegador una sola vez.")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

cds_client = cdsapi.Client(url="https://cds.climate.copernicus.eu/api", key=CDS_API_KEY)
sample_hours = pd.date_range(SAMPLE_START_UTC, periods=N_HOURS, freq="h", tz="UTC")
query_hours = pd.date_range(sample_hours.min() - pd.Timedelta(days=3), sample_hours.max(), freq="h", tz="UTC")
area = [
    CENTER_LAT + AREA_PADDING_DEG,
    CENTER_LON - AREA_PADDING_DEG,
    CENTER_LAT - AREA_PADDING_DEG,
    CENTER_LON + AREA_PADDING_DEG,
]
print(f"Se compararán {len(sample_hours)} horas; se consultan {len(query_hours)} para calcular acumulados.")

## 3. Extracción CDS

In [ ]:
cds_started = perf_counter()
cds_downloaded_files = 0
cds_api_hours = 0

CDS_VARIABLES = [
    "2m_temperature", "2m_dewpoint_temperature",
    "10m_u_component_of_wind", "10m_v_component_of_wind",
    "surface_solar_radiation_downwards", "total_precipitation",
]
CDS_SHORT = ["t2m", "d2m", "u10", "v10", "ssrd", "tp"]

groups = defaultdict(list)
for timestamp in query_hours:
    groups[(timestamp.year, timestamp.month)].append(timestamp)

cds_paths = []
for (year, month), timestamps in sorted(groups.items()):
    days = sorted({timestamp.day for timestamp in timestamps})
    target = OUTPUT_DIR / f"cds_{year}_{month:02d}_{min(days):02d}-{max(days):02d}.nc"
    cds_api_hours += len(days) * 24
    if OVERWRITE or not target.exists():
        partial = target.with_suffix(".nc.part")
        partial.unlink(missing_ok=True)
        cds_client.retrieve("reanalysis-era5-land", {
            "variable": CDS_VARIABLES,
            "year": str(year), "month": f"{month:02d}",
            "day": [f"{day:02d}" for day in days],
            "time": [f"{hour:02d}:00" for hour in range(24)],
            "area": area,
            "data_format": "netcdf", "download_format": "unarchived",
        }, str(partial))
        partial.replace(target)
        cds_downloaded_files += 1
    cds_paths.append(target)

frames = []
for path in cds_paths:
    ds = xr.open_dataset(path)
    frame = ds[CDS_SHORT].to_dataframe().reset_index()
    time_col = next(name for name in ["valid_time", "time"] if name in frame)
    frames.append(frame.rename(columns={time_col: "datetime_utc"}))
    ds.close()
cds_raw = pd.concat(frames, ignore_index=True)
cds_raw["datetime_utc"] = pd.to_datetime(cds_raw["datetime_utc"], utc=True)
cds_raw = cds_raw.sort_values(["latitude", "longitude", "datetime_utc"])
cds_parts = []
for _, group in cds_raw.groupby(["latitude", "longitude"], sort=False):
    part = deaccumulate_era5_land(
        group.rename(columns={"datetime_utc": "era5_dt_utc"}), ["ssrd", "tp"]
    ).rename(columns={"era5_dt_utc": "datetime_utc"})
    cds_parts.append(part)
cds_hourly = pd.concat(cds_parts, ignore_index=True)
cds_elapsed_s = perf_counter() - cds_started
print("CDS:", cds_hourly.shape, "| píxeles:", cds_hourly[["latitude", "longitude"]].drop_duplicates().shape[0])
print(f"Tiempo CDS: {cds_elapsed_s:.2f} s | archivos descargados: {cds_downloaded_files}")

## 4. Extracción equivalente desde Earth Engine

In [ ]:
gee_started = perf_counter()

GEE_BANDS = [
    "temperature_2m", "dewpoint_temperature_2m",
    "u_component_of_wind_10m", "v_component_of_wind_10m",
    "surface_solar_radiation_downwards_hourly", "total_precipitation_hourly",
]
GEE_TO_SHORT = dict(zip(GEE_BANDS, CDS_SHORT))

grid = cds_hourly[["latitude", "longitude"]].drop_duplicates().sort_values(["latitude", "longitude"])
points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([row.longitude, row.latitude]), {
        "latitude": float(row.latitude), "longitude": float(row.longitude)
    })
    for row in grid.itertuples()
])
collection = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY")

def compute_gee_features(expression, *, max_attempts=5):
    """Evalúa una sola agregación y reintenta únicamente el límite temporal 429."""
    for attempt in range(max_attempts):
        try:
            return ee.data.computeFeatures({
                "expression": expression,
                "fileFormat": "PANDAS_DATAFRAME",
                "pageSize": 1000,
                "workloadTag": "era5-cds-gee-validation",
            })
        except Exception as exc:
            is_concurrency_limit = "Too many concurrent aggregations" in str(exc)
            if not is_concurrency_limit or attempt == max_attempts - 1:
                raise
            wait_seconds = 2 ** (attempt + 1)
            print(f"Earth Engine ocupado; reintento en {wait_seconds} s.")
            sleep(wait_seconds)

stacked_images = []
stacked_band_groups = []
for position, timestamp in enumerate(query_hours):
    start = timestamp.isoformat()
    end = (timestamp + pd.Timedelta(hours=1)).isoformat()
    image = ee.Image(collection.filterDate(start, end).first())
    band_names = [f"h{position:03d}_{short}" for short in CDS_SHORT]
    stacked_images.append(image.select(GEE_BANDS).rename(band_names))
    stacked_band_groups.append(band_names)

stacked = ee.Image.cat(*stacked_images)
samples = stacked.reduceRegions(
    collection=points, reducer=ee.Reducer.first(), scale=11132
)
print(f"Earth Engine: {len(query_hours)} horas apiladas en una agregación.")
gee_wide = compute_gee_features(samples)
gee_parts = []
for timestamp, band_names in zip(query_hours, stacked_band_groups):
    part = gee_wide[["latitude", "longitude", *band_names]].copy()
    part = part.rename(columns=dict(zip(band_names, CDS_SHORT)))
    part.insert(0, "datetime_utc", timestamp)
    gee_parts.append(part)
gee_hourly = pd.concat(gee_parts, ignore_index=True)
gee_hourly = gee_hourly[["datetime_utc", "latitude", "longitude", *CDS_SHORT]]
gee_elapsed_s = perf_counter() - gee_started
print("Earth Engine:", gee_hourly.shape)
print(f"Tiempo Earth Engine: {gee_elapsed_s:.2f} s")

## 5. Comparación bruta y derivada

In [ ]:
keys = ["datetime_utc", "latitude", "longitude"]
comparison = cds_hourly.merge(gee_hourly, on=keys, suffixes=("_cds", "_gee"), how="outer", indicator=True)
comparison = comparison[comparison["datetime_utc"].isin(sample_hours)].copy()

RAW_TOLERANCE = {
    "t2m": 0.05, "d2m": 0.05, "u10": 0.05, "v10": 0.05,
    "ssrd": 7200.0,  # 2 W/m² durante una hora
    "tp": 0.0001,    # 0,1 mm
}
detail = []
for variable in CDS_SHORT:
    part = comparison[keys + ["_merge", f"{variable}_cds", f"{variable}_gee"]].copy()
    part = part.rename(columns={f"{variable}_cds": "value_cds", f"{variable}_gee": "value_gee"})
    part["variable"] = variable
    part["absolute_difference"] = (part.value_cds - part.value_gee).abs()
    denominator = part[["value_cds", "value_gee"]].abs().max(axis=1).replace(0, np.nan)
    part["relative_difference_percent"] = 100 * part.absolute_difference / denominator
    detail.append(part)
detail = pd.concat(detail, ignore_index=True)

raw_rows = []
for variable, group in detail.groupby("variable"):
    valid = group.dropna(subset=["value_cds", "value_gee"])
    differences = valid.value_gee - valid.value_cds
    raw_rows.append({
        "variable": variable, "compared": len(valid),
        "mae": differences.abs().mean(), "mean_bias": differences.mean(),
        "max_absolute_error": differences.abs().max(),
        "correlation": valid.value_cds.corr(valid.value_gee),
    })
raw_summary = pd.DataFrame(raw_rows)
raw_summary["tolerance"] = raw_summary.variable.map(RAW_TOLERANCE)
raw_summary["passes"] = raw_summary.max_absolute_error <= raw_summary.tolerance

def regional_derived(frame):
    regional = frame.groupby("datetime_utc", as_index=False)[CDS_SHORT].mean()
    regional = regional.rename(columns={"datetime_utc": "era5_dt_utc"})
    return derive_era5(regional)

cds_derived, cds_daily = regional_derived(cds_hourly)
gee_derived, gee_daily = regional_derived(gee_hourly)
FINAL_TOLERANCE = {
    "Tair_C": 0.05, "RH": 0.5, "Rsol_Wm2": 2.0,
    "wind_speed": 0.05, "rain_1d": 0.1, "rain_3d": 0.1,
    "rain_3d_log": 0.01, "is_rainy": 0.0,
}
final_rows = []
for variable in ["Tair_C", "RH", "Rsol_Wm2", "wind_speed"]:
    joined = cds_derived[["era5_dt_utc", variable]].merge(
        gee_derived[["era5_dt_utc", variable]], on="era5_dt_utc", suffixes=("_cds", "_gee")
    )
    joined = joined[joined.era5_dt_utc.isin(sample_hours)]
    errors = (joined[f"{variable}_cds"] - joined[f"{variable}_gee"]).abs()
    final_rows.append({
        "variable": variable, "compared": len(errors), "mae": errors.mean(),
        "mean_bias": (joined[f"{variable}_gee"] - joined[f"{variable}_cds"]).mean(),
        "max_absolute_error": errors.max(),
        "correlation": joined[f"{variable}_cds"].corr(joined[f"{variable}_gee"]),
    })
sample_days = pd.DatetimeIndex(sample_hours.floor("D").unique())
for variable in ["rain_1d", "rain_3d", "rain_3d_log", "is_rainy"]:
    joined = cds_daily[["day", variable]].merge(gee_daily[["day", variable]], on="day", suffixes=("_cds", "_gee"))
    joined = joined[joined.day.isin(sample_days)]
    errors = (joined[f"{variable}_cds"] - joined[f"{variable}_gee"]).abs()
    final_rows.append({
        "variable": variable, "compared": len(errors), "mae": errors.mean(),
        "mean_bias": (joined[f"{variable}_gee"] - joined[f"{variable}_cds"]).mean(),
        "max_absolute_error": errors.max(),
        "correlation": joined[f"{variable}_cds"].corr(joined[f"{variable}_gee"]),
    })
final_summary = pd.DataFrame(final_rows)
final_summary["tolerance"] = final_summary.variable.map(FINAL_TOLERANCE)
final_summary["passes"] = final_summary.max_absolute_error <= final_summary.tolerance

display(raw_summary)
display(final_summary)

## 6. Informe y decisión

In [ ]:
comparable_values = N_HOURS * len(grid) * len(CDS_SHORT)
performance = pd.DataFrame([
    {
        "source": "CDS",
        "elapsed_seconds": cds_elapsed_s,
        "api_hours_requested": cds_api_hours,
        "rows_received": len(cds_hourly),
        "comparable_values": comparable_values,
        "execution_mode": "red" if cds_downloaded_files else "caché",
    },
    {
        "source": "Earth Engine",
        "elapsed_seconds": gee_elapsed_s,
        "api_hours_requested": len(query_hours),
        "rows_received": len(gee_hourly),
        "comparable_values": comparable_values,
        "execution_mode": "red",
    },
])
performance["rows_per_second"] = performance["rows_received"] / performance["elapsed_seconds"].clip(lower=1e-9)
performance["seconds_per_1000_comparable_values"] = (
    performance["elapsed_seconds"] / comparable_values * 1000
)
fastest = performance.loc[performance["elapsed_seconds"].idxmin()]
slowest = performance.loc[performance["elapsed_seconds"].idxmax()]
speed_factor = slowest.elapsed_seconds / max(fastest.elapsed_seconds, 1e-9)
timing_verdict = f"{fastest.source} fue {speed_factor:.2f} veces más rápido en esta ejecución."

coverage_ok = comparison["_merge"].eq("both").all()
values_complete = detail[["value_cds", "value_gee"]].notna().all(axis=None)
all_pass = bool(coverage_ok and values_complete and raw_summary.passes.all() and final_summary.passes.all())
verdict = (
    "APTO: se recomienda usar Earth Engine; reproduce CDS dentro de las tolerancias definidas."
    if all_pass else
    "NO APTO TODAVÍA: conserva CDS y revisa las variables que superan la tolerancia o las correspondencias ausentes."
)

detail.to_csv(OUTPUT_DIR / "comparacion_detallada.csv", index=False)
raw_summary.to_csv(OUTPUT_DIR / "resumen_variables_brutas.csv", index=False)
final_summary.to_csv(OUTPUT_DIR / "resumen_variables_finales.csv", index=False)
performance.to_csv(OUTPUT_DIR / "rendimiento_fuentes.csv", index=False)
report = [
    "VALIDACIÓN ERA5-LAND: CDS FRENTE A EARTH ENGINE",
    f"Muestra: {N_HOURS} horas desde {SAMPLE_START_UTC}",
    f"Cobertura temporal y espacial completa: {'sí' if coverage_ok else 'no'}",
    f"Valores completos: {'sí' if values_complete else 'no'}",
    "",
    "RENDIMIENTO OBSERVADO",
    *[
        f"- {row.source}: {row.elapsed_seconds:.2f} s; {row.rows_received} filas; "
        f"{row.rows_per_second:.1f} filas/s; modo {row.execution_mode}."
        for row in performance.itertuples()
    ],
    timing_verdict,
    "",
    verdict,
    "",
    "VARIABLES FINALES",
    *[
        f"- {row.variable}: MAE={row.mae:.6g}; máximo={row.max_absolute_error:.6g}; "
        f"tolerancia={row.tolerance:.6g}; {'OK' if row.passes else 'REVISAR'}"
        for row in final_summary.itertuples()
    ],
    "",
    "Una discrepancia exclusiva en precipitación o radiación suele indicar una convención de acumulación distinta.",
]
(OUTPUT_DIR / "informe_comparacion.txt").write_text("\n".join(report) + "\n", encoding="utf-8")
print(verdict)
print(timing_verdict)
display(performance)
print("Resultados:", OUTPUT_DIR.resolve())